# Persian Emotional TTS
## نوت‌بوک مرحله‌ای پروژه در Google Colab

این نوت‌بوک برای تولید گفتار فارسی با کنترل لحن و احساس، بر اساس نقشه راه پنج‌فازی پروژه تنظیم شده است.

- **محیط اجرا:** Google Colab
- **ذخیره‌سازی پایدار:** Google Drive
- **مسیر پروژه:** `/content/drive/MyDrive/Persian_Emotional_TTS`
- **وضعیت فعلی:** زیرساخت و کتابخانه‌ها آماده‌اند؛ گام ۱.۳ با FLEURS و گام ۱.۴ با G2P و alignment در همین نوت‌بوک ادامه دارند.

> سلول‌ها را به ترتیب و فعلاً فقط تا پایان گام ۱.۳ اجرا کن. سایر گام‌ها فعلاً فقط ساختار و محل مشخص دارند و انجام‌شده محسوب نمی‌شوند.

## نقشه راه پروژه

### فاز ۱: زیرساخت، ابزارها و آماده‌سازی داده
- [x] **۱.۱:** راه‌اندازی Colab، اتصال Drive و ساختار پوشه‌بندی
- [x] **۱.۲:** نصب کتابخانه‌های کلیدی AI، پردازش صوت و مدل‌سازی
- [ ] **۱.۳:** جمع‌آوری پیکره داده‌های داستانی و صوتی
- [ ] **۱.۴:** پیش‌پردازش متن فارسی (G2P) و Force Alignment
- [ ] **۱.۵:** برچسب‌گذاری احساسی اولیه (Pseudo-Labeling)

### فاز ۲: ماژول درک متن و تحلیل احساس
- [ ] **۲.۱:** انتخاب مدل زبانی سبک فارسی
- [ ] **۲.۲:** استخراج بردارهای Emotion/Prosody از متن

### فاز ۳: مدل آکوستیک و ووکودر
- [ ] **۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS
- [ ] **۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی احساس
- [ ] **۳.۳:** اتصال BigVGAN برای تولید صوت

### فاز ۴: اجرای بلادرنگ و استریم
- [ ] **۴.۱:** Sentence/Clause Chunking
- [ ] **۴.۲:** Async Pipeline برای کاهش تأخیر
- [ ] **۴.۳:** ONNX و WebSocket Audio Streaming

### فاز ۵: ارزیابی و بهبود
- [ ] **۵.۱:** ارزیابی انسانی و MOS
- [ ] **۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال

## قبل از اجرا

1. نوت‌بوک را در [Google Colab](https://colab.research.google.com/) باز کن.
2. از مسیر **Runtime → Change runtime type → Hardware accelerator** گزینه **GPU** را انتخاب کن.
3. سلول‌ها را از بالا به پایین اجرا کن.
4. هنگام اتصال Google Drive، دسترسی لازم را تأیید کن.
5. پس از اجرای هر گام، خروجی سلول پایانی همان گام را برای بررسی نگه دار.

# فاز ۱ — زیرساخت، ابزارها و آماده‌سازی داده
## گام ۱.۱ — بررسی محیط، اتصال Drive و ساختار پروژه

### سلول ۱: بررسی Python، PyTorch و GPU

این سلول نصب جدیدی انجام نمی‌دهد و فقط وضعیت runtime فعلی Colab را گزارش می‌کند.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA runtime:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: Select a GPU from Runtime > Change runtime type.")
except ImportError:
    print("PyTorch is not installed; it will be handled in step 1.2.")

### سلول ۲: اتصال Google Drive

در اولین اجرا، Colab پنجره اجازه دسترسی به Drive را نمایش می‌دهد. حساب موردنظر را انتخاب و دسترسی را تأیید کن.

In [ ]:
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

assert Path("/content/drive/MyDrive").is_dir(), "Google Drive mount failed."
print("Google Drive is ready.")

### سلول ۳: ساخت پوشه‌ها و فایل پیکربندی

اجرای این سلول امن و قابل تکرار است؛ فایل‌های صوتی یا مدل‌های موجود حذف نمی‌شوند.

In [ ]:
import json
from pathlib import Path

PROJECT_NAME = "Persian_Emotional_TTS"
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
DIRECTORIES = (
    "data/raw",
    "data/processed",
    "models/acoustic",
    "models/emotion_extractor",
    "models/vocoder",
    "configs",
)

for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)
    print(f"Ready: {relative_path}/")

project_config = {
    "project_name": PROJECT_NAME,
    "project_root": str(PROJECT_ROOT),
    "directories": list(DIRECTORIES),
}
config_path = PROJECT_ROOT / "configs" / "project_config.json"
config_path.write_text(
    json.dumps(project_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


### سلول ۴: اعتبارسنجی ساختار پروژه

این سلول وجود همه پوشه‌ها و فایل پیکربندی را بررسی می‌کند.

In [ ]:
missing_directories = [
    relative_path
    for relative_path in DIRECTORIES
    if not (PROJECT_ROOT / relative_path).is_dir()
]
assert not missing_directories, f"Missing directories: {missing_directories}"
assert config_path.is_file(), "Project configuration was not created."

print("Project root:", PROJECT_ROOT)
print("Structure verification: PASSED")
print("Step 1.1 setup: VERIFIED")

## گام ۱.۲ — نصب کتابخانه‌های کلیدی

### سلول ۵: نصب یا تکمیل کتابخانه‌ها

Colab معمولاً PyTorch متناسب با CUDA خودش را از قبل دارد؛ بنابراین نسخه CUDA آن را بی‌دلیل تغییر نمی‌دهیم. اگر PyTorch/Torchaudio/Torchvision موجود نباشند، این سلول آن‌ها را هم نصب می‌کند.

کتابخانه‌های اختصاصی Matcha-TTS و BigVGAN عمداً در این مرحله نصب نمی‌شوند تا در فاز سوم، نسخه‌های سازگار با معماری پروژه انتخاب شوند.

In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
]

for package in ("torch", "torchaudio", "torchvision"):
    if importlib.util.find_spec(package) is None:
        packages.append(package)

print("Installing/checking:", ", ".join(packages))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)
print("Package installation completed.")

### سلول ۶: تست import و نسخه‌های کتابخانه‌ها

اجرای این سلول تایید می‌کند که همه کتابخانه‌های موردنیاز گام ۱.۲ بدون خطا import می‌شوند و در صورت وجود GPU، یک تست ساده CUDA نیز انجام می‌شود.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

packages_to_verify = (
    "torch",
    "torchaudio",
    "torchvision",
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
)

failed_imports = []
for package in packages_to_verify:
    try:
        importlib.import_module(package)
        try:
            package_version = version(package)
        except PackageNotFoundError:
            package_version = "version unavailable"
        print(f"OK  {package:<14} {package_version}")
    except Exception as exc:
        failed_imports.append(package)
        print(f"FAIL {package}: {type(exc).__name__}: {exc}")

if failed_imports:
    raise RuntimeError(
        "Step 1.2 is not complete. Failed imports: " + ", ".join(failed_imports)
    )

import torch

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    test_tensor = torch.rand(1024, device="cuda")
    print("CUDA tensor test: PASSED; mean =", round(test_tensor.mean().item(), 4))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Imports passed, but no GPU is active in this runtime.")

print("\nEnvironment verification: PASSED")

## گام ۱.۲ — اجرا و تأیید شد

کتابخانه‌های موردنیاز نصب و import آن‌ها بررسی شده است. اکنون سلول‌های گام ۱.۳ را ادامه بده. داده‌های پروژه همچنان روی Google Drive باقی می‌مانند.

## گام ۱.۳ — دانلود و آماده‌سازی پیکره پایه فارسی

در این گام از **Google FLEURS** با زبان و کانفیگ `fa_ir` استفاده می‌کنیم. این دیتاست برای راه‌اندازی و تست اولیه مدل آکوستیک مناسب است و نیازی به آپلود دستی فایل‌ها ندارد.

**نکته‌های مهم:**

- FLEURS دیتاست گفتار خوانده‌شده و پایه است؛ **دیتاست داستان‌گویی احساسی نیست**. داده داستانی و احساسی در مراحل بعد به ساختار پروژه اضافه می‌شود.
- صوت اصلی FLEURS معمولاً **۱۶۰۰۰ هرتز** است. خروجی این نوت‌بوک برای یکسان‌سازی خط پردازش، به **WAV مونو با ۲۲۰۵۰ هرتز** تبدیل می‌شود؛ resample کردن به‌خودی‌خود جزئیات ازدست‌رفته را برنمی‌گرداند.
- متن خام و علائم آن تا گام ۱.۴ حفظ می‌شود؛ در این مرحله نرمال‌سازی متن فارسی انجام نمی‌دهیم.
- برای دانلود، Colab باید به اینترنت و Hugging Face دسترسی داشته باشد.

### سلول ۷: تنظیمات و دانلود FLEURS فارسی

برای یک تست سریع می‌توانی مقدار `MAX_SAMPLES_PER_SPLIT` را مثلاً `20` بگذاری. مقدار `None` یعنی پردازش همه نمونه‌های splitهای انتخاب‌شده.

این سلول فقط دیتاست را دانلود و در cache کتابخانه `datasets` قرار می‌دهد؛ تبدیل فایل‌ها در سلول بعد انجام می‌شود.

In [ ]:
import json
from pathlib import Path

from datasets import load_dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
assert Path("/content/drive/MyDrive").is_dir(), "Run the Google Drive mounting cell first."
assert PROJECT_ROOT.is_dir(), "Project root not found; run the step 1.1 setup cells first."

DATASET_NAME = "google/fleurs"
DATASET_CONFIG = "fa_ir"
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "fleurs_fa_ir"
SPLITS = ("train", "validation", "test")
MAX_SAMPLES_PER_SPLIT = None  # Use an integer such as 20 for a quick test.
OUTPUT_SAMPLE_RATE = 22050
OVERWRITE_EXISTING = False

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ("audio", "transcripts"):
    (DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)

print(f"Loading {DATASET_NAME} ({DATASET_CONFIG})...")
fleurs = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(fleurs)

for split in SPLITS:
    assert split in fleurs, f"Split {split!r} is not available in FLEURS."

run_config = {
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "splits": list(SPLITS),
    "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
    "output_sample_rate": OUTPUT_SAMPLE_RATE,
    "output_format": "WAV PCM_16 mono",
    "text_field_preference": ["raw_transcription", "transcription"],
}
(DATASET_ROOT / "dataset_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Dataset download and configuration: READY")

### سلول ۸: تبدیل صوت‌ها و ساخت فایل‌های متن و metadata

خروجی در مسیر زیر ذخیره می‌شود:

```text
data/raw/fleurs_fa_ir/
├── audio/{train,validation,test}/
├── transcripts/{train,validation,test}/
├── metadata.csv
└── dataset_config.json
```

فایل‌ها نام‌های یکتا و قابل تکرار دارند. اجرای دوباره سلول، فایل‌های موجود را به‌صورت پیش‌فرض بازنویسی نمی‌کند. برای بازنویسی، `OVERWRITE_EXISTING = True` را در سلول قبل تنظیم کن.

In [ ]:
import csv
import re
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm


def as_mono(audio_array):
    """Convert decoded audio to a one-dimensional float32 mono array."""
    array = np.asarray(audio_array, dtype=np.float32)
    if array.ndim == 1:
        return array
    if array.ndim == 2:
        # Handle either (channels, samples) or (samples, channels).
        channel_axis = 0 if array.shape[0] <= array.shape[1] else 1
        return array.mean(axis=channel_axis, dtype=np.float32)
    raise ValueError(f"Unexpected audio shape: {array.shape}")


rows = []
for split in SPLITS:
    split_data = fleurs[split]
    if MAX_SAMPLES_PER_SPLIT is not None:
        sample_count = min(MAX_SAMPLES_PER_SPLIT, len(split_data))
        split_data = split_data.select(range(sample_count))

    for index, row in enumerate(tqdm(split_data, desc=f"Preparing {split}")):
        source_id = str(row.get("id", index))
        safe_source_id = re.sub(r"[^A-Za-z0-9_-]+", "_", source_id)
        utterance_id = f"fleurs_fa_ir_{split}_{index:06d}_{safe_source_id}"

        audio_path = DATASET_ROOT / "audio" / split / f"{utterance_id}.wav"
        transcript_path = DATASET_ROOT / "transcripts" / split / f"{utterance_id}.txt"
        audio_path.parent.mkdir(parents=True, exist_ok=True)
        transcript_path.parent.mkdir(parents=True, exist_ok=True)

        text = str(row.get("raw_transcription") or row.get("transcription") or "").strip()
        if not text:
            raise ValueError(f"Empty transcription for {split} sample {index}.")

        audio_data = row["audio"]
        if not audio_data or "array" not in audio_data:
            raise ValueError(f"Decoded audio is unavailable for {utterance_id}.")
        source_sample_rate = int(audio_data["sampling_rate"])
        audio_array = as_mono(audio_data["array"])
        if audio_array.size == 0 or source_sample_rate <= 0:
            raise ValueError(f"Empty or invalid audio for {utterance_id}.")

        if source_sample_rate != OUTPUT_SAMPLE_RATE:
            audio_array = librosa.resample(
                audio_array,
                orig_sr=source_sample_rate,
                target_sr=OUTPUT_SAMPLE_RATE,
                res_type="kaiser_best",
            )
        audio_array = np.clip(audio_array, -1.0, 1.0)

        if OVERWRITE_EXISTING or not audio_path.exists():
            sf.write(str(audio_path), audio_array, OUTPUT_SAMPLE_RATE, subtype="PCM_16")
        if OVERWRITE_EXISTING or not transcript_path.exists():
            transcript_path.write_text(text + "\n", encoding="utf-8")

        rows.append({
            "utterance_id": utterance_id,
            "audio_path": audio_path.relative_to(PROJECT_ROOT).as_posix(),
            "transcript_path": transcript_path.relative_to(PROJECT_ROOT).as_posix(),
            "text": text,
            "speaker_id": str(row.get("speaker_id", "unknown")),
            "source": DATASET_NAME,
            "source_config": DATASET_CONFIG,
            "source_id": source_id,
            "split": split,
            "duration_seconds": round(len(audio_array) / OUTPUT_SAMPLE_RATE, 3),
            "sample_rate": OUTPUT_SAMPLE_RATE,
            "source_sample_rate": source_sample_rate,
            "channels": 1,
        })

metadata_path = DATASET_ROOT / "metadata.csv"
fieldnames = (
    "utterance_id", "audio_path", "transcript_path", "text", "speaker_id",
    "source", "source_config", "source_id", "split", "duration_seconds",
    "sample_rate", "source_sample_rate", "channels",
)
with metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)

print("Prepared utterances:", len(rows))
print("Total duration (minutes):", round(sum(row["duration_seconds"] for row in rows) / 60, 2))
print("Metadata:", metadata_path)
print("FLEURS audio and transcript export: COMPLETE")

### سلول ۹: اعتبارسنجی مستقل خروجی FLEURS

این سلول فایل `metadata.csv` را از روی Drive می‌خواند و وجود صوت و متن، مونو بودن، نرخ ۲۲۰۵۰ هرتز و غیرخالی بودن متن‌ها را بررسی می‌کند. نتیجه آن را برای ورود به گام ۱.۴ ارسال کن.

In [ ]:
import csv
from collections import Counter

import soundfile as sf

assert metadata_path.is_file(), "metadata.csv was not created; rerun the export cell."
with metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    manifest_rows = list(csv.DictReader(csv_file))

assert manifest_rows, "The FLEURS manifest is empty."
validation_errors = []
for row in manifest_rows:
    audio_path = PROJECT_ROOT / row["audio_path"]
    transcript_path = PROJECT_ROOT / row["transcript_path"]
    try:
        assert audio_path.is_file(), f"Missing audio: {audio_path}"
        assert transcript_path.is_file(), f"Missing transcript: {transcript_path}"
        info = sf.info(str(audio_path))
        assert info.samplerate == OUTPUT_SAMPLE_RATE, (
            f"Unexpected sample rate for {row['utterance_id']}: {info.samplerate}"
        )
        assert info.channels == 1, f"Not mono: {row['utterance_id']}"
        assert info.frames > 0, f"Empty audio: {row['utterance_id']}"
        transcript = transcript_path.read_text(encoding="utf-8").strip()
        assert transcript, f"Empty transcript: {row['utterance_id']}"
        assert transcript == row["text"].strip(), f"Text mismatch: {row['utterance_id']}"
    except Exception as exc:
        validation_errors.append(f"{row['utterance_id']}: {exc}")

if validation_errors:
    raise ValueError("FLEURS validation failed:\n" + "\n".join(validation_errors[:20]))

split_counts = Counter(row["split"] for row in manifest_rows)
total_seconds = sum(float(row["duration_seconds"]) for row in manifest_rows)
print("Validated utterances:", len(manifest_rows))
print("Rows per split:", dict(split_counts))
print("Total duration (minutes):", round(total_seconds / 60, 2))
print("Output sample rate:", OUTPUT_SAMPLE_RATE)
print("FLEURS fa_ir preparation: PASSED")

## گام ۱.۳ — پیکره پایه آماده شد

پس از اجرای موفق سلول‌های ۷ تا ۹، دیتاست پایه FLEURS فارسی در Drive قرار دارد و می‌توانیم از آن برای تست خط پیش‌پردازش و آموزش اولیه استفاده کنیم.

این خروجی به‌تنهایی مدل را داستان‌گو یا احساسی نمی‌کند. در ادامه به داده داستانی و احساسی، یا ضبط اختصاصی با متن هماهنگ، نیاز داریم. گام بعدی **۱.۴: نرمال‌سازی متن فارسی، G2P و Force Alignment** است.

## گام ۱.۴ — پیش‌پردازش متن فارسی، G2P و Force Alignment

در این گام دو خروجی جدا ساخته می‌شود:

1. **متن نرمال‌شده و IPA:** با `hazm` و `eSpeak-NG` از طریق `phonemizer`.
2. **همگام‌سازی کلمه‌ای:** با `stable-ts` و استفاده از متن مرجع هر فایل، نه تولید متن جدید توسط ASR.

G2P در این مرحله یک baseline قابل اجرا برای فارسی است و بعداً می‌توان آن را با مدل تخصصی‌تر فارسی جایگزین کرد. کیفیت alignment نیز باید روی نمونه‌هایی از داده بررسی شود؛ اجرای بدون خطا به‌تنهایی اثبات دقت تمام مرزهای کلمات نیست.

### سلول ۱۰: نصب ابزارهای G2P و alignment

این سلول `espeak-ng` و `ffmpeg` را در محیط Colab نصب می‌کند و سپس کتابخانه‌های Python لازم را نصب می‌نماید. اجرای آن فقط در Colab لازم است.

In [ ]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng", "ffmpeg"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "hazm", "phonemizer", "stable-ts"],
    check=True,
)
print("G2P and alignment dependencies: READY")

### سلول ۱۱: نرمال‌سازی متن و تولید IPA

متن خام `metadata.csv` تغییر نمی‌کند. خروجی این سلول در `data/processed/fleurs_fa_ir/metadata_text.csv` ذخیره می‌شود و شامل `normalized_text` و `phonemes` است.

In [ ]:
import csv
import json
from pathlib import Path

import hazm
from phonemizer import phonemize
from phonemizer.separator import Separator

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
RAW_METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "fleurs_fa_ir" / "metadata.csv"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fleurs_fa_ir"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

assert RAW_METADATA_PATH.is_file(), "Run the FLEURS preparation cells first."
with RAW_METADATA_PATH.open("r", encoding="utf-8", newline="") as csv_file:
    raw_rows = list(csv.DictReader(csv_file))
assert raw_rows, "The FLEURS metadata is empty."

normalizer = hazm.Normalizer()
phone_separator = Separator(phone=" ", word=" | ")
text_rows = []
g2p_errors = []

for row in raw_rows:
    try:
        normalized_text = " ".join(normalizer.normalize(row["text"]).split())
        if not normalized_text:
            raise ValueError("Normalized text is empty.")
        phonemes = phonemize(
            normalized_text,
            language="fa",
            backend="espeak",
            separator=phone_separator,
            strip=True,
            preserve_punctuation=False,
            njobs=1,
        ).strip()
        if not phonemes:
            raise ValueError("G2P returned an empty phoneme sequence.")
        output_row = dict(row)
        output_row["normalized_text"] = normalized_text
        output_row["phonemes"] = phonemes
        text_rows.append(output_row)
    except Exception as exc:
        g2p_errors.append({"utterance_id": row["utterance_id"], "error": str(exc)})

if g2p_errors:
    (PROCESSED_ROOT / "g2p_errors.json").write_text(
        json.dumps(g2p_errors, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    raise RuntimeError(f"G2P failed for {len(g2p_errors)} rows; see g2p_errors.json")

text_metadata_path = PROCESSED_ROOT / "metadata_text.csv"
with text_metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(text_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(text_rows)

print("G2P processed utterances:", len(text_rows))
print("Text metadata:", text_metadata_path)
print("Sample normalized text:", text_rows[0]["normalized_text"])
print("Sample IPA:", text_rows[0]["phonemes"])
print("Persian text preprocessing: PASSED")

### سلول ۱۲: بارگذاری مدل alignment

مدل `base` برای اولین اجرا سبک‌تر است. اگر کیفیت مرزهای کلمات کافی نبود، می‌توان بعداً `small` را امتحان کرد؛ این تغییر سرعت و مصرف حافظه را بیشتر می‌کند.

`ALIGNMENT_LIMIT = 20` فقط برای Smoke Test است. برای پردازش همه ردیف‌ها، پس از موفقیت تست آن را به `None` تغییر بده.

In [ ]:
import torch
import stable_whisper

ALIGNMENT_MODEL_NAME = "base"
ALIGNMENT_LIMIT = 20  # Smoke test only; set to None for the full manifest.
OVERWRITE_ALIGNMENT = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Alignment device:", DEVICE)
print("Loading alignment model:", ALIGNMENT_MODEL_NAME)
alignment_model = stable_whisper.load_model(ALIGNMENT_MODEL_NAME, device=DEVICE)
print("Alignment model: READY")

### سلول ۱۳: اجرای Force Alignment روی متن مرجع

برای هر نمونه یک فایل JSON با timestamp کلمات ساخته می‌شود. فایل‌های موجود در اجرای بعدی خوانده می‌شوند، بنابراین بعد از تست ۲۰ نمونه‌ای می‌توانی با `ALIGNMENT_LIMIT = None` ادامه بدهی.

اگر متن و صوت مطابقت نداشته باشند یا مدل نتواند کلمات را align کند، خطا ثبت و گزارش می‌شود؛ نباید خروجی مشکوک را بدون بررسی برای آموزش استفاده کرد.

In [ ]:
import csv
import json
from tqdm.auto import tqdm

alignment_root = PROCESSED_ROOT / "alignments"
alignment_root.mkdir(parents=True, exist_ok=True)
aligned_metadata_path = PROCESSED_ROOT / "metadata_aligned.csv"

alignment_input_rows = text_rows if ALIGNMENT_LIMIT is None else text_rows[:ALIGNMENT_LIMIT]
aligned_rows = []
alignment_errors = []

for row in tqdm(alignment_input_rows, desc="Force aligning"):
    utterance_id = row["utterance_id"]
    alignment_path = alignment_root / f"{utterance_id}.json"
    try:
        if alignment_path.exists() and not OVERWRITE_ALIGNMENT:
            alignment_payload = json.loads(alignment_path.read_text(encoding="utf-8"))
        else:
            result = alignment_model.align(
                str(PROJECT_ROOT / row["audio_path"]),
                row["text"],
                language="fa",
                regroup=False,
            )
            result_dict = result.to_dict()
            words = []
            for segment in result_dict.get("segments", []):
                for word in segment.get("words", []) or []:
                    word_text = str(word.get("word", "")).strip()
                    if not word_text:
                        continue
                    words.append({
                        "word": word_text,
                        "start": float(word.get("start") or 0.0),
                        "end": float(word.get("end") or 0.0),
                    })
            if not words:
                raise ValueError("No word timestamps were returned.")
            alignment_payload = {
                "utterance_id": utterance_id,
                "audio_path": row["audio_path"],
                "text": row["text"],
                "normalized_text": row["normalized_text"],
                "phonemes": row["phonemes"],
                "language": "fa",
                "alignment_model": ALIGNMENT_MODEL_NAME,
                "segments": result_dict.get("segments", []),
                "words": words,
            }
            alignment_path.write_text(
                json.dumps(alignment_payload, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

        words = alignment_payload.get("words", [])
        if not words:
            raise ValueError("Alignment JSON contains no words.")
        output_row = dict(row)
        output_row["alignment_path"] = alignment_path.relative_to(PROJECT_ROOT).as_posix()
        output_row["alignment_word_count"] = len(words)
        aligned_rows.append(output_row)
    except Exception as exc:
        alignment_errors.append({"utterance_id": utterance_id, "error": str(exc)})

if alignment_errors:
    (PROCESSED_ROOT / "alignment_errors.json").write_text(
        json.dumps(alignment_errors, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    raise RuntimeError(
        f"Alignment failed for {len(alignment_errors)} rows; see alignment_errors.json"
    )

with aligned_metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(aligned_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(aligned_rows)

print("Aligned utterances:", len(aligned_rows))
print("Aligned metadata:", aligned_metadata_path)
print("Alignment JSON directory:", alignment_root)
print("Force alignment export: COMPLETE")

### سلول ۱۴: اعتبارسنجی timestampها و خروجی alignment

این سلول وجود فایل JSON، تعداد کلمات و معتبر بودن بازه‌های زمانی را بررسی می‌کند. اگر `ALIGNMENT_LIMIT` هنوز ۲۰ باشد، نتیجه فقط اعتبارسنجی یک زیرمجموعه آزمایشی است، نه پردازش کامل دیتاست.

In [ ]:
import csv
import json

assert aligned_metadata_path.is_file(), "Aligned metadata was not created; rerun the alignment cell."
with aligned_metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    aligned_manifest = list(csv.DictReader(csv_file))

expected_count = len(alignment_input_rows)
assert len(aligned_manifest) == expected_count, "Aligned row count does not match the requested subset."
alignment_validation_errors = []
word_count = 0

for row in aligned_manifest:
    try:
        payload_path = PROJECT_ROOT / row["alignment_path"]
        payload = json.loads(payload_path.read_text(encoding="utf-8"))
        words = payload["words"]
        assert words, f"No words in {row['utterance_id']}"
        assert len(words) == int(row["alignment_word_count"])
        previous_start = -1.0
        for word in words:
            start = float(word["start"])
            end = float(word["end"])
            assert start >= 0 and end > start, f"Invalid word span: {word}"
            assert start >= previous_start, f"Non-monotonic word timestamps: {row['utterance_id']}"
            previous_start = start
            word_count += 1
    except Exception as exc:
        alignment_validation_errors.append(f"{row['utterance_id']}: {exc}")

if alignment_validation_errors:
    raise ValueError("Alignment validation failed:\n" + "\n".join(alignment_validation_errors[:20]))

print("Validated aligned utterances:", len(aligned_manifest))
print("Validated word timestamps:", word_count)
print("Alignment subset limit:", ALIGNMENT_LIMIT)
print("Alignment output verification: PASSED")

## گام ۱.۴ — نتیجه و ادامه

با اجرای موفق سلول‌های ۱۰ تا ۱۴، مسیر G2P و alignment تست شده است. اگر مقدار `ALIGNMENT_LIMIT` را روی ۲۰ گذاشته‌ای، برای پردازش کامل دیتاست آن را به `None` تغییر بده و سلول‌های ۱۳ و ۱۴ را دوباره اجرا کن.

خروجی‌های اصلی:

```text
data/processed/fleurs_fa_ir/metadata_text.csv
data/processed/fleurs_fa_ir/metadata_aligned.csv
data/processed/fleurs_fa_ir/alignments/
```

قبل از استفاده برای آموزش، چند فایل JSON و صوت متناظر را به‌صورت شنیداری و بصری بررسی کن. گام بعدی **۱.۵: Pseudo-Labeling احساسی** است.

## گام بعدی فاز ۱ — هنوز پیاده‌سازی نشده است

### گام ۱.۵: Pseudo-Labeling احساسی
زیربخش‌ها: انتخاب دسته‌های احساس، تولید برچسب اولیه و کنترل کیفیت برچسب‌ها.

# فاز ۲ — Context & Emotion Extractor
## وضعیت: در انتظار تکمیل فاز ۱

- **گام ۲.۱:** انتخاب مدل زبانی سبک فارسی، مانند ParsBERT یا نسخه کوچک Qwen.
- **گام ۲.۲:** Fine-tuning برای استخراج Emotion/Prosody Vectors از متن داستان.

# فاز ۳ — Acoustic Model & Vocoder
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS.
- **گام ۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی بردار احساس.
- **گام ۳.۳:** اتصال BigVGAN برای تولید صوت باکیفیت و حفظ جزئیات عاطفی.

# فاز ۴ — Real-Time Pipeline
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۴.۱:** Sentence/Clause Chunking هوشمند.
- **گام ۴.۲:** طراحی Async Pipeline برای کاهش تأخیر اولیه.
- **گام ۴.۳:** صادرات ONNX و استریم صوت با WebSocket.

# فاز ۵ — Evaluation & Fine-Tuning
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۵.۱:** ارزیابی ذهنی و انسانی کیفیت لحن‌ها با MOS.
- **گام ۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال‌های احتمالی.